## Proposed Bus Routes - BTO

In [151]:
import folium
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import LineString, Point
from mrt_map import get_mrt_map

In [152]:
df_202407 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202407.csv")
df_202408 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202408.csv")
df_202409 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202409.csv")
combined_df = pd.concat([df_202407, df_202408, df_202409], ignore_index=True)

# Filter for DAY_TYPE == 'WEEKDAY' and TIME_PER_HOUR for peak hours [7, 8, 9, 10, 17, 18, 19, 20]
filtered_df = combined_df[(combined_df['DAY_TYPE'] == 'WEEKENDS/HOLIDAY') & 
                          (combined_df['TIME_PER_HOUR'].isin([9, 10, 11, 12, 17, 18, 19, 20, 21]))]
summarised_df = pd.DataFrame(columns=['PT_CODE', 'TOTAL_VOLUME'])
# Group by 'PT_CODE' and calculate the sum of tap-in and tap-out volumes
grouped = filtered_df.groupby('PT_CODE').agg(
    TOTAL_TAP_IN_VOLUME=('TOTAL_TAP_IN_VOLUME', 'sum'),
    TOTAL_TAP_OUT_VOLUME=('TOTAL_TAP_OUT_VOLUME', 'sum')).reset_index()
# Create a new column for the total volume (sum of tap-in and tap-out volumes)
grouped['TOTAL_VOLUME'] = grouped['TOTAL_TAP_IN_VOLUME'] + grouped['TOTAL_TAP_OUT_VOLUME']
summarised_df['PT_CODE'] = grouped['PT_CODE']
summarised_df['TOTAL_VOLUME'] = grouped['TOTAL_VOLUME']
summarised_df = summarised_df.sort_values(by='TOTAL_VOLUME', ascending=False)
trunkroutes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")
trunkroutes_grouped = trunkroutes.groupby('BusStopCode').first().reset_index()
# Merge summarised_df with trunkroutes based on PT_CODE == BusStopCode
merged_df = pd.merge(summarised_df, trunkroutes_grouped[['BusStopCode', 'Description', 'Latitude', 'Longitude','Direction']], 
                     left_on='PT_CODE', right_on='BusStopCode', how='left')
merged_df = merged_df.drop(columns=['BusStopCode'])
merged_df.to_csv('location_popular_bus_Stops.csv', index=False)
filtered_df = merged_df[~merged_df['Description'].str.contains('Stn|Int', na=False)].reset_index().drop('index', axis=1)

In [153]:
# Create a function to scale the size of the marker based on TOTAL_VOLUME
def scale_marker_size(volume, min_size=5, max_size=15):
    volume_range = merged_df['TOTAL_VOLUME'].max() - merged_df['TOTAL_VOLUME'].min()
    if volume_range == 0:
        return min_size  # Avoid division by zero
    scaled_size = ((volume - merged_df['TOTAL_VOLUME'].min()) / volume_range) * (max_size - min_size) + min_size
    return scaled_size

## Add BTO Locations

In [154]:
singapore = get_mrt_map()
## BTOs with over 1,000 units
taman_jurong_skyline = [1.3270289195127982, 103.72582148080623]
tanjong_rhu = [1.2996481809089269, 103.88025809169478]
teban_breeze = [1.3218378113117057, 103.74476401645141]
chencharu_hills = [1.419134809710128, 103.82527849062635]
marsiling_peak = [1.444461323034268, 103.77610683781455]
woodgrove_edge = [1.4290005301535544, 103.78480587301898]


# Add a red circle marker for the bto locations
folium.CircleMarker(
    location=taman_jurong_skyline,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Taman Jurong Skyline", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=tanjong_rhu,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Tanjong Rhu Riverfront I & II", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=teban_breeze,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Teban Breeze", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=chencharu_hills,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Chenchura Hills", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=marsiling_peak,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Marsiling Peak", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=woodgrove_edge,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("WoodGrove Edge", max_width=100)
).add_to(singapore)


# Display the map
singapore

/Users/krystal/Documents/GitHub/DSA4264/DSA4264/venv/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geo

## Plot bus routes for buses servicing each BTO 

In [155]:
trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

nearest_stop_codes = {
    'taman_jurong_skyline': 21769,
    'tanjong_rhu': 90051,
    'teban_breeze': 20261,
    'chencharu_hills': 57069,
    'marsiling_peak': 46121,
    'woodgrove_edge': 46229
}

bto_colors = {
    'taman_jurong_skyline': 'orange',
    'tanjong_rhu': 'green',
    'teban_breeze': 'red',
    'chencharu_hills': 'purple',
    'marsiling_peak': 'brown',
    'woodgrove_edge': 'pink'
}

filtered_routes = {}

for bto, stop_code in nearest_stop_codes.items():
    # Find the bus services stopping at the nearest bus stop
    bto_services = trunk_bus_routes[trunk_bus_routes['BusStopCode'] == stop_code]['ServiceNo'].unique()
    
    # Filter the bus_routes DataFrame to include only those services
    filtered_routes[bto] = trunk_bus_routes[trunk_bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service, and add only the markers for the stops they visit
for bto, routes in filtered_routes.items():
    color = bto_colors[bto]  # Get the color for the current BTO location
    for service_no in routes['ServiceNo'].unique():
        # Extract route points for each service
        route_points = routes[routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
        
        # Draw the route on the map
        folium.PolyLine(
            locations=route_points,
            color=color,  # Use the color corresponding to the BTO location
            weight=3,
            opacity=0.6,
            popup=f"Service {service_no} - {bto}"
        ).add_to(singapore)
        
        # Add markers for each stop along this route
        for idx, row in routes[routes['ServiceNo'] == service_no].iterrows():
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
            ).add_to(singapore)

# Plot first 100 rows on the singapore_mrt map
for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(singapore)
singapore


# Display the map with filtered routes and stops
singapore


## Tanjong Rhu

In [156]:
tanjong_rhu_map = get_mrt_map()

tanjong_rhu = [1.2996481809089269, 103.88025809169478]

folium.CircleMarker(
    location=tanjong_rhu,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Tanjong Rhu Riverfront I & II", max_width=100)
).add_to(tanjong_rhu_map)


trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")


nearest_stop_code = 90051  

bto_color = 'green'


bto_services = trunk_bus_routes[trunk_bus_routes['BusStopCode'] == nearest_stop_code]['ServiceNo'].unique()


filtered_routes = trunk_bus_routes[trunk_bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service, and add only the markers for the stops they visit
for service_no in filtered_routes['ServiceNo'].unique():
    # Extract route points for each service
    route_points = filtered_routes[filtered_routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
    
    # Draw the route on the map
    folium.PolyLine(
        locations=route_points,
        color=bto_color,  # Use the color for Tanjong Rhu
        weight=3,
        opacity=0.6,
        popup=f"Service {service_no} - Tanjong Rhu"
    ).add_to(tanjong_rhu_map)
    
    # Add markers for each stop along this route
    for idx, row in filtered_routes[filtered_routes['ServiceNo'] == service_no].iterrows():
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=5,
            color=bto_color,
            fill=True,
            fill_color=bto_color,
            fill_opacity=0.7,
            popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
        ).add_to(tanjong_rhu_map)

for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(tanjong_rhu_map)
tanjong_rhu_map

# Display the map with Tanjong Rhu routes and stops
tanjong_rhu_map


/Users/krystal/Documents/GitHub/DSA4264/DSA4264/venv/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geo

## Tanjong Rhu Riverfront Proposed Route

In [157]:
tanjong_rhu_riverfront_route = ['90061', '90051', '1039', '7518', '7419', '7319','7111', '7031', '40011', '9037', '9219','9179', '9022','9037','5039','3031']

trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(tanjong_rhu_riverfront_route)]
filtered_stops = filtered_stops.set_index('BusStopCode').loc[tanjong_rhu_riverfront_route].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(tanjong_rhu_map)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(tanjong_rhu_map)


bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(tanjong_rhu_map)
tanjong_rhu_map

## Taman Jurong Skyline

In [160]:
taman_jurong_skyline_map = get_mrt_map()

# Define the Woodgrove Edge location
taman_jurong_skyline = [1.3270289195127982, 103.72582148080623]

# Add a red circle marker for Woodgrove Edge
folium.CircleMarker(
    location=taman_jurong_skyline,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Taman Jurong Skyline", max_width=100)
).add_to(taman_jurong_skyline_map)

# Load bus routes data
bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

# Define the nearest bus stop code for Taman Jurong Skyline
nearest_stop_code = 21769  # Taman Jurong Skyline bus stop code

# Define the color for Taman Jurong Skyline routes
bto_color = 'orange'

# Find the bus services stopping at the nearest bus stop
bto_services = bus_routes[bus_routes['BusStopCode'] == nearest_stop_code]['ServiceNo'].unique()

# Filter the bus_routes DataFrame to include only those services
filtered_routes = bus_routes[bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service, and add only the markers for the stops they visit
for service_no in filtered_routes['ServiceNo'].unique():
    # Extract route points for each service
    route_points = filtered_routes[filtered_routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
    
    # Draw the route on the map
    folium.PolyLine(
        locations=route_points,
        color=bto_color,  # Use the color for Taman Jurong Skyline
        weight=3,
        opacity=0.6,
        popup=f"Service {service_no} - Taman Jurong Skyline"
    ).add_to(taman_jurong_skyline_map)
    
    # Add markers for each stop along this route
    for idx, row in filtered_routes[filtered_routes['ServiceNo'] == service_no].iterrows():
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=5,
            color=bto_color,
            fill=True,
            fill_color=bto_color,
            fill_opacity=0.7,
            popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
        ).add_to(taman_jurong_skyline_map)

for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(taman_jurong_skyline_map)

# Display the map with Taman Jurong Skyline routes and stops
taman_jurong_skyline_map


/Users/krystal/Documents/GitHub/DSA4264/DSA4264/venv/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geo

## Taman Jurong Skyline Proposed Route

In [162]:
taman_jurong_skyline_route = ['28099', '21449', '21429', '28401', '28511', '28461', '28301', '43411', '43189', '1039', '9022', '9179', '9219', '9037', '1039', '7518', '40011', '7419', '7111']

trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(taman_jurong_skyline_route)]
filtered_stops = filtered_stops.set_index('BusStopCode').loc[taman_jurong_skyline_route].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(taman_jurong_skyline_map)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(taman_jurong_skyline_map)


bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(taman_jurong_skyline_map)
taman_jurong_skyline_map